In [0]:

# ============================================================
# CELDA 1. VALIDACIÓN DE CALIDAD DE SILVER
# ============================================================

# Revisamos las tablas Silver de vuelos y clima.
# El objetivo es detectar problemas antes de Gold
# y de entrenar los modelos de Machine Learning.
#
# Esta celda solo consulta información.
# No elimina ni modifica registros.

from pyspark.sql import functions as F

TABLA_VUELOS = "workspace.silver_flights.flights_clean"
TABLA_CLIMA = "workspace.silver_flights.weather_clean"

vuelos = spark.table(TABLA_VUELOS)
clima = spark.table(TABLA_CLIMA)

# ------------------------------------------------------------
# 1. VALIDAR VUELOS
# ------------------------------------------------------------

print("=" * 55)
print("1. CALIDAD DE SILVER - VUELOS")
print("=" * 55)

columnas_vuelos = [
    "flight_date",
    "origin",
    "dest",
    "airline",
    "crs_dep_time",
    "dep_delay",
    "delayed_15"
]

# Contamos nulos en las columnas esenciales.
nulos_vuelos = vuelos.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in columnas_vuelos
]).first().asDict()

print("Nulos en columnas esenciales:", nulos_vuelos)

# Comprobamos que no haya cancelaciones en esta tabla.
cancelados = vuelos.filter(
    F.col("cancelled") != 0
).count()

print("Vuelos cancelados incluidos:", cancelados)

# Revisamos si la etiqueta corresponde a DEPDELAY.
etiquetas_incorrectas = vuelos.filter(
    F.col("delayed_15") !=
    F.when(F.col("dep_delay") >= 15, 1).otherwise(0)
).count()

print("Etiquetas delayed_15 incorrectas:", etiquetas_incorrectas)

# Buscamos filas completamente repetidas.
# Excluimos tail_number: a veces se encuentra vacío
# y queremos revisar si existen registros idénticos
# en los demás campos.
columnas_comparacion = [
    "flight_date", "origin", "dest", "airline",
    "crs_dep_time", "dep_delay"
]

duplicados_potenciales = (
    vuelos
    .groupBy(*columnas_comparacion)
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Grupos de posibles vuelos duplicados:",
    duplicados_potenciales.count()
)

# ------------------------------------------------------------
# 2. VALIDAR CLIMA
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("2. CALIDAD DE SILVER - CLIMA")
print("=" * 55)

columnas_clima = [
    "airport",
    "weather_time_utc",
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

nulos_clima = clima.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in columnas_clima
]).first().asDict()

print("Nulos en columnas meteorológicas:", nulos_clima)

# Cada aeropuerto debe tener una sola fila por hora,
# modelo y horizonte de pronóstico.
duplicados_clima = (
    clima
    .groupBy(
        "airport",
        "weather_time_utc",
        "source_model",
        "forecast_lead_hours"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Grupos duplicados en clima:", duplicados_clima)

# Validamos rangos físicamente razonables.
# Estos límites sirven para detectar anomalías,
# no representan normas meteorológicas oficiales.
anomalias_clima = clima.filter(
    (F.col("precipitation") < 0) |
    (F.col("wind_speed_10m") < 0) |
    (~F.col("relative_humidity_2m").between(0, 100)) |
    (~F.col("cloud_cover").between(0, 100)) |
    (F.col("pressure_msl") <= 0)
).count()

print("Registros con rangos meteorológicos anómalos:",
      anomalias_clima)

# ------------------------------------------------------------
# 3. RESULTADO GENERAL
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RESUMEN DE CALIDAD SILVER")
print("=" * 55)

print("Registros de vuelos:", f"{vuelos.count():,}")
print("Registros de clima:", f"{clima.count():,}")
print("Posibles duplicados de vuelos:",
      duplicados_potenciales.count())

if (
    all(v == 0 for v in nulos_vuelos.values())
    and cancelados == 0
    and etiquetas_incorrectas == 0
    and all(v == 0 for v in nulos_clima.values())
    and duplicados_clima == 0
    and anomalias_clima == 0
):
    print("VALIDACIÓN BÁSICA EXITOSA.")
    print(
        "Los posibles duplicados de vuelos requieren "
        "revisión adicional antes de eliminarlos."
    )
else:
    print("REVISAR: se encontraron problemas de calidad.")


In [0]:

# ============================================================
# CELDA 2. IDENTIFICAR ANOMALÍAS METEOROLÓGICAS
# ============================================================

# Revisamos qué variable provoca las 371 alertas.
# No eliminamos registros ni modificamos Silver.
#
# Los límites se utilizan para detectar valores
# inesperados que necesitamos investigar.

from pyspark.sql import functions as F

clima = spark.table("workspace.silver_flights.weather_clean")

# Definimos las condiciones que utilizamos
# en la validación anterior.
condiciones = {
    "Precipitación negativa":
        F.col("precipitation") < 0,

    "Viento negativo":
        F.col("wind_speed_10m") < 0,

    "Humedad fuera de 0-100":
        ~F.col("relative_humidity_2m").between(0, 100),

    "Nubosidad fuera de 0-100":
        ~F.col("cloud_cover").between(0, 100),

    "Presión menor o igual a cero":
        F.col("pressure_msl") <= 0
}

# Contamos las anomalías por condición
# utilizando una sola agregación de Spark.
resultado = clima.agg(*[
    F.sum(
        F.when(condicion, 1).otherwise(0)
    ).alias(nombre)
    for nombre, condicion in condiciones.items()
]).first().asDict()

print("ANOMALÍAS POR VARIABLE")
print("-" * 45)

for variable, cantidad in resultado.items():
    print(f"{variable}: {cantidad:,}")

# Mostramos los rangos reales de las variables.
print("\nRANGOS OBSERVADOS")

columnas = [
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

display(
    clima.agg(*[
        expresion
        for columna in columnas
        for expresion in [
            F.min(columna).alias(f"{columna}_min"),
            F.max(columna).alias(f"{columna}_max")
        ]
    ])
)


In [0]:

# ============================================================
# CELDA 3. REVISAR VALORES ANÓMALOS DE NUBOSIDAD
# ============================================================

# Identificamos los valores negativos de nubosidad
# y comprobamos en qué aeropuertos se presentan.
#
# Esta consulta no modifica Silver.

from pyspark.sql import functions as F

clima = spark.table("workspace.silver_flights.weather_clean")

anomalias = clima.filter(
    F.col("cloud_cover") < 0
)

print("Registros con nubosidad negativa:", anomalias.count())

print("\nVALORES NEGATIVOS ENCONTRADOS")
display(
    anomalias
    .groupBy("cloud_cover")
    .count()
    .orderBy("cloud_cover")
)

print("\nAEROPUERTOS CON ANOMALÍAS")
display(
    anomalias
    .groupBy("airport")
    .count()
    .orderBy(F.desc("count"))
)

print("\nEJEMPLOS DE REGISTROS")
display(
    anomalias.select(
        "airport",
        "weather_time_utc",
        "cloud_cover"
    ).limit(10)
)


In [0]:

# ============================================================
# CELDA 4. REVISIÓN COMPLETA DE NUBOSIDAD
# ============================================================

# Revisamos todos los valores de nubosidad fuera
# del rango válido de 0% a 100%.
# No modificamos ni eliminamos registros.

from pyspark.sql import functions as F

clima = spark.table("workspace.silver_flights.weather_clean")

# Mostramos cada valor fuera del rango y su frecuencia.
anomalias_nubosidad = (
    clima
    .filter(
        (F.col("cloud_cover") < 0) |
        (F.col("cloud_cover") > 100)
    )
    .groupBy("cloud_cover")
    .count()
    .orderBy("cloud_cover")
)

print("VALORES ANÓMALOS DE NUBOSIDAD")
display(anomalias_nubosidad)

print(
    "Total de registros anómalos:",
    clima.filter(
        (F.col("cloud_cover") < 0) |
        (F.col("cloud_cover") > 100)
    ).count()
)


### Conclusiones de validación Silver

Se revisaron las tablas Silver de vuelos y clima para identificar valores faltantes, duplicados y problemas de consistencia.

La tabla de vuelos contiene 8,655,359 registros, sin valores nulos en las columnas esenciales, sin vuelos cancelados y con la variable objetivo `delayed_15` correctamente calculada. Se identificaron 179 grupos de posibles duplicados, que requieren una revisión adicional antes de eliminar registros.

En la tabla meteorológica se validaron 504,000 registros, sin valores nulos ni duplicados por aeropuerto y hora. Sin embargo, se encontraron 371 registros con nubosidad fuera del rango de 0% a 100%: 120 con valor −1 y 251 con valor 101.

Para el entrenamiento de los modelos se propone ajustar estos valores al rango permitido de 0 a 100, conservando los datos originales en Bronze. Las demás variables meteorológicas no presentaron anomalías con los criterios utilizados.